# Reinforcement Learning

# 4. Online control

This notebook presents the **online control** of an agent by SARSA and Q-learning.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
from model import TicTacToe, Nim, ConnectFour
from agent import Agent, OnlineControl
from dynamic import ValueIteration

## To do

* Complete the class ``SARSA`` and test it on Tic-Tac-Toe.
* Complete the class ``QLearning`` and test it on Tic-Tac-Toe.
* Compare these algorithms on Tic-Tac-Toe (play first) and Nim (play second), using a random adversary, then a perfect adversary. Comment your results.
* Test these algorithms on Connect 4 against a random adversary. Comment your results.

## SARSA

In [3]:
class SARSA(OnlineControl):
    """Online control by SARSA."""
        
    def update_values(self, state=None, horizon=100, epsilon=0.5):
        """Learn the action-value function online."""
        self.model.reset(state)
        state = self.model.state
        if not self.model.is_terminal(state):
            action = self.randomize_best_action(state, epsilon=epsilon)
            for t in range(horizon):
                code = self.model.encode(state)
                self.action_count[code][action] += 1
                reward, stop = self.model.step(action)
                # to be modified (get sample gain)
                # begin
                if not stop:
                    next_state = self.model.state
                    next_code = self.model.encode(next_state)
                    next_action = self.randomize_best_action(next_state, epsilon=epsilon)
                    gain = reward + self.gamma * self.action_value[next_code][next_action]
                else:
                    gain = reward
                # end
                diff = gain - self.action_value[code][action]
                count = self.action_count[code][action]
                self.action_value[code][action] += diff / count
                if stop:
                    break
                # to be modified (update state and action)
                # begin
                state = next_state
                action = next_action
                # end

## Q-learning

In [4]:
class QLearning(OnlineControl):
    """Online control by Q-learning."""
        
    def update_values(self, state=None, horizon=100, epsilon=0.5):
        """Learn the action-value function online."""
        self.model.reset(state)
        state = self.model.state
        # to be completed
        if not self.model.is_terminal(state):
            for t in range(horizon):
                code = self.model.encode(state)
                action = self.randomize_best_action(state, epsilon=epsilon)
                self.action_count[code][action] += 1
                reward, stop = self.model.step(action)

                if not stop:
                    next_state = self.model.state
                    next_code = self.model.encode(next_state)
                    next_action = self.randomize_best_action(next_state, epsilon=0)
                    gain = reward + self.gamma * self.action_value[next_code][next_action]
                else:
                    gain = reward

                diff = gain - self.action_value[code][action]
                count = self.action_count[code][action]
                self.action_value[code][action] += diff / count

                if stop:
                    break

                state = next_state
                    

## To do

In [5]:
def run_training(Game, Algo, adversary_policy, play_first=True, n_games=1000, epsilon=0.5, step=None, verbose=False):
    game = Game(adversary_policy=adversary_policy, play_first=play_first)
    algo = Algo(game, policy='random')
    gains = []

    if verbose:
        print("Game: {}".format(Game.__name__))
        if isinstance(adversary_policy, str):
            print("Adversary: {}".format(adversary_policy))
        else:
            print("Adversary: costum")
        print("Learning algorithm: {}".format(Algo.__name__), end='\n\n')

    for t in range(n_games):
        algo.update_values(epsilon=epsilon)
        if step and ((t+1)%step == 0 or t == 0):
            policy_improved = algo.get_policy()
            agent = Agent(game, policy=policy_improved)
            gain = np.mean(agent.get_gains())
            gains.append(gain)
            if verbose:
                print("Gain after {} games: {}".format(t+1, gain))

    policy_improved = algo.get_policy()
    agent = Agent(game, policy=policy_improved)
        
    results = dict(zip(*np.unique(agent.get_gains(), return_counts=True)))
    gain = np.mean(agent.get_gains())

    if verbose:
        print() if step else None
        print("--Final policy--")
        print("Losses: {}".format(results[-1] if -1 in results else 0))
        print("Draws: {}".format(results[0] if 0 in results else 0))
        print("Wins: {}".format(results[1] if 1 in results else 0))
        
    return gains if step else gain

In [6]:
def plot_gains(data_dict, game, adversary_policy, num_episodes, step):
    # X-axis values
    games = np.arange(0, num_episodes + 1, step)

    for key, values in data_dict.items():
        plt.plot(games, values, label=f"{key}", marker="o")

    # Labels, title, and legend
    plt.xlabel("Games")
    plt.ylabel("Gain")
    plt.title(f"{game}: {adversary_policy.title()} adversary")
    plt.legend(loc="best")
    plt.ylim(-1, 1)

### Random agent on TicTacToe

In [7]:
Game = TicTacToe
game = Game()
agent = Agent(game)

results = dict(zip(*np.unique(agent.get_gains(), return_counts=True)))

In [ ]:
print("Losses: {}".format(results[-1] if -1 in results else 0))
print("Draws: {}".format(results[0] if 0 in results else 0))
print("Wins: {}".format(results[1] if 1 in results else 0))

### SARSA on TicTacToe

In [ ]:
gain = run_training(Game, SARSA, adversary_policy='random', play_first=True, n_games=1000, epsilon=0.1, verbose=True)

### QLearning on TicTacToe

In [ ]:
gain = run_training(Game, QLearning, adversary_policy='random', play_first=True, n_games=1000, epsilon=0.1, verbose=True)

## TicTacToe: SARSA

In [11]:
epsilons = [0.05, 0.1, 0.5]
n_games = 4000
step = n_games/10

In [12]:
game = TicTacToe()

algo = ValueIteration(game)
policy, perfect_tictactoe = algo.get_perfect_players()

### Random player

In [13]:
sarsa_tictactoe_random_gains = {epsilon: run_training(TicTacToe,
                                                      SARSA,
                                                      adversary_policy='random',
                                                      play_first=True,
                                                      n_games=n_games,
                                                      epsilon=epsilon,
                                                      step=step)
                                                      
                                                      for epsilon in epsilons}

### Perfect player

In [14]:
sarsa_tictactoe_perfect_gains = {epsilon: run_training(TicTacToe,
                                                       SARSA,
                                                       adversary_policy=perfect_tictactoe,
                                                       play_first=True,
                                                       n_games=n_games,
                                                       epsilon=epsilon,
                                                       step=step)
                                                       
                                                       for epsilon in epsilons}

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Plot gains
plt.sca(axes[0])
plot_gains(sarsa_tictactoe_random_gains, 'TicTacToe', 'random', n_games, step)

plt.sca(axes[1])
plot_gains(sarsa_tictactoe_perfect_gains, 'TicTacToe', 'perfect', n_games, step)

plt.tight_layout()
plt.show()

## TicTacToe: QLearning

In [16]:
epsilons = [0.05, 0.1, 0.5]
n_games = 4000
step = n_games/10

### Random player

In [17]:
qlearning_tictactoe_random_gains = {epsilon: run_training(TicTacToe,
                                                          QLearning,
                                                          adversary_policy='random',
                                                          play_first=True,
                                                          n_games=n_games,
                                                          epsilon=epsilon,
                                                          step=step)
                                                          
                                                          for epsilon in epsilons}

### Perfect player

In [18]:
qlearning_tictactoe_perfect_gains = {epsilon: run_training(TicTacToe,
                                                           QLearning,
                                                           adversary_policy=perfect_tictactoe,
                                                           play_first=True,
                                                           n_games=n_games,
                                                           epsilon=epsilon,
                                                           step=step)
                                                           
                                                           for epsilon in epsilons}

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Plot gains
plt.sca(axes[0])
plot_gains(qlearning_tictactoe_random_gains, 'TicTacToe', 'random', n_games, step)

plt.sca(axes[1])
plot_gains(qlearning_tictactoe_perfect_gains, 'TicTacToe', 'perfect', n_games, step)

plt.tight_layout()
plt.show()

### Interpretation: TicTacToe (SARSA vs QLearning)

SARSA and QLearning have similar performance against a random policy adversary, both getting near-perfect gains. The choice of epsilon has minimal impact in this scenario. However, against a perfect adversary, QLearning shows more stability in gain improvement across games, while SARSA sometimes shows oscillations in gain during some runs. Both algorithms converge toward optimal players (gain of 0), which is the best outcome possible against a perfect player in TicTacToe.

## Nim: SARSA

In [20]:
epsilons = [0.05, 0.1, 0.5]
n_games = 4000
step = n_games/10

In [21]:
game = Nim(play_first=False)

algo = ValueIteration(game)
policy, perfect_nim = algo.get_perfect_players()

### Random player

In [22]:
sarsa_nim_random_gains = {epsilon: run_training(Nim,
                                                SARSA,
                                                adversary_policy='random',
                                                play_first=False,
                                                n_games=n_games,
                                                epsilon=epsilon,
                                                step=step)
                                                
                                                for epsilon in epsilons}

### Perfect player

In [23]:
sarsa_nim_perfect_gains = {epsilon: run_training(Nim,
                                                 SARSA,
                                                 adversary_policy=perfect_nim,
                                                 play_first=False,
                                                 n_games=n_games,
                                                 epsilon=epsilon,
                                                 step=step)
                                                 
                                                 for epsilon in epsilons}

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Plot gains
plt.sca(axes[0])
plot_gains(sarsa_nim_random_gains, 'Nim', 'random', n_games, step)

plt.sca(axes[1])
plot_gains(sarsa_nim_perfect_gains, 'Nim', 'perfect', n_games, step)

plt.tight_layout()
plt.show()

## Nim: QLearning

In [25]:
epsilons = [0.05, 0.1, 0.5]
n_games = 4000
step = n_games/10

### Random player

In [26]:
qlearning_nim_random_gains = {epsilon: run_training(Nim,
                                                    QLearning,
                                                    adversary_policy='random',
                                                    play_first=False,
                                                    n_games=n_games,
                                                    epsilon=epsilon,
                                                    step=step)

                                                    for epsilon in epsilons}

### Perfect player

In [27]:
qlearning_nim_perfect_gains = {epsilon: run_training(Nim,
                                                     QLearning,
                                                     adversary_policy=perfect_nim,
                                                     play_first=False,
                                                     n_games=n_games,
                                                     epsilon=epsilon,
                                                     step=step)
                                                     
                                                     for epsilon in epsilons}

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Plot gains
plt.sca(axes[0])
plot_gains(qlearning_nim_random_gains, 'Nim', 'random', n_games, step)

plt.sca(axes[1])
plot_gains(qlearning_nim_perfect_gains, 'Nim', 'perfect', n_games, step)

plt.tight_layout()
plt.show()

### Interpretation: Nim (SARSA vs QLearning)

Similar to TicTacToe, SARSA and QLearning perform similar against a random adversary, converging close to the optimal action-value function to secure a high win-rate by 2,000 games. Against a perfect player, the learning process requires approximately 2x as many games for the agent to get consistent wins with gains similar to those against a random adversary. The choice of epsilon impacts the learning process against a perfect player but not against a random adversary. An exploration rate of 0.5 hinders progress for both SARSA and QLearning; a smaller epsilon is more effective due to the relatively small state space, where suboptimal actions slow down learning.

## SARSA vs QLearning

In [ ]:
tictactoe_perfect_gains = {'sarsa': sarsa_tictactoe_perfect_gains[0.05],
                           'qlearning': qlearning_tictactoe_perfect_gains[0.05]}

nim_perfect_gains = {'sarsa': sarsa_nim_perfect_gains[0.05],
                     'qlearning': qlearning_nim_perfect_gains[0.05]}

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Plot gains
plt.sca(axes[0])
plot_gains(tictactoe_perfect_gains, 'TicTacToe', 'perfect', n_games, step)

plt.sca(axes[1])
plot_gains(nim_perfect_gains, 'Nim', 'perfect', n_games, step)

plt.tight_layout()
plt.show()

### Interpretation: SARSA vs QLearning

The plots show the gain progression for SARSA and QLearning in two games, TicTacToe and Nim, using an exploration rate of 0.05 against a perfect player. As discussed previously, SARSA's learning process often oscillates while improving gains, whereas QLearning steadily increases the gain until reaching 0, the highest achievable against a perfect player (consistent ties).

In Nim, both algorithms perform well, eventually reaching a nearly optimal gain (0.8). This outcome aligns with expectations, as playing second in this Nim setup guarantees a win when executed perfectly.

## ConnectFour

In [30]:
epsilons = [0.1]
n_games = 3000
step = n_games/10

### SARSA

In [31]:
sarsa_connectfour_random_gains = {epsilon: run_training(ConnectFour,
                                                        SARSA,
                                                        adversary_policy='random',
                                                        play_first=True,
                                                        n_games=n_games,
                                                        epsilon=epsilon,
                                                        step=step)
                                                        
                                                        for epsilon in epsilons}

### QLearning

In [32]:
qlearning_connectfour_random_gains = {epsilon: run_training(ConnectFour,
                                                            QLearning,
                                                            adversary_policy='random',
                                                            play_first=True,
                                                            n_games=n_games,
                                                            epsilon=epsilon,
                                                            step=step)

                                                            for epsilon in epsilons}

In [ ]:
connectfour_random_gains = {'sarsa': sarsa_connectfour_random_gains[0.1],
                           'qlearning': qlearning_connectfour_random_gains[0.1]}

plot_gains(connectfour_random_gains, 'ConnectFour', 'random', n_games, step)

plt.tight_layout()
plt.show()

### Interpretation: ConnectFour

The plot shows both learning algorithms (SARSA and QLearning) with an exploration rate of 0.1. The agent is not getting any better at the game against a random adversary. With more games (5,000). But this takes a long time to run so only 3000 games are done here. This performance is because of the difference in the state space size between TicTacToe, Nim and ConnectFour.